# Rebaseline Tuning — Iteration 2 (Move-First)
Notebook này dùng để tuning cấu hình move features + XGBoost theo rule notebook-first.

In [4]:
from pathlib import Path
import sys

import polars as pl
import pandas as pd

# Đảm bảo import được package src khi chạy từ notebook ở thư mục con
cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.feature_config import FeatureConfig
from src.feature_engineering import FeaturePipeline
from src.rebaseline import RebaselineRunner, RebaselineConfig

In [6]:
# Chuẩn bị sample input
sample_dir = repo_root / 'data' / 'features' / 'smoke'
sample_dir.mkdir(parents=True, exist_ok=True)
train_src = repo_root / 'data' / 'processed' / 'lichess_2025-12_ml.parquet'
val_src = repo_root / 'data' / 'processed' / 'lichess_2026-01_ml.parquet'
train_sample = sample_dir / 'train_sample_input.parquet'
val_sample = sample_dir / 'val_sample_input.parquet'
pl.scan_parquet(str(train_src)).head(3000).collect().write_parquet(train_sample)
pl.scan_parquet(str(val_src)).head(3000).collect().write_parquet(val_sample)
train_sample, val_sample

(PosixPath('/home/sakana/Code/PTIT/MMDs/MMD-G2/data/features/smoke/train_sample_input.parquet'),
 PosixPath('/home/sakana/Code/PTIT/MMDs/MMD-G2/data/features/smoke/val_sample_input.parquet'))

In [7]:
# Grid cấu hình cần so sánh
grid = [
    {'name': 'cfg_a_1to2_df2', 'n_ply': 15, 'ng': (1,2), 'max_feat': 500, 'min_df': 2, 'max_df': 0.95, 'est': 120, 'depth': 6, 'lr': 0.06},
    {'name': 'cfg_b_2to3_df1', 'n_ply': 15, 'ng': (2,3), 'max_feat': 600, 'min_df': 1, 'max_df': 0.98, 'est': 140, 'depth': 6, 'lr': 0.06},
    {'name': 'cfg_c_1to2_df1_deeper', 'n_ply': 18, 'ng': (1,2), 'max_feat': 700, 'min_df': 1, 'max_df': 0.98, 'est': 180, 'depth': 7, 'lr': 0.05},
]
grid

[{'name': 'cfg_a_1to2_df2',
  'n_ply': 15,
  'ng': (1, 2),
  'max_feat': 500,
  'min_df': 2,
  'max_df': 0.95,
  'est': 120,
  'depth': 6,
  'lr': 0.06},
 {'name': 'cfg_b_2to3_df1',
  'n_ply': 15,
  'ng': (2, 3),
  'max_feat': 600,
  'min_df': 1,
  'max_df': 0.98,
  'est': 140,
  'depth': 6,
  'lr': 0.06},
 {'name': 'cfg_c_1to2_df1_deeper',
  'n_ply': 18,
  'ng': (1, 2),
  'max_feat': 700,
  'min_df': 1,
  'max_df': 0.98,
  'est': 180,
  'depth': 7,
  'lr': 0.05}]

In [8]:
# Chạy tuning loop
rows = []
for c in grid:
    cfg = FeatureConfig(
        batch_size=1000,
        n_ply=c['n_ply'],
        tfidf_max_features=c['max_feat'],
        tfidf_ngram_min=c['ng'][0],
        tfidf_ngram_max=c['ng'][1],
        tfidf_min_df=c['min_df'],
        tfidf_max_df=c['max_df'],
        train_features_file=sample_dir / f"train_features_{c['name']}.parquet",
        val_features_file=sample_dir / f"val_features_{c['name']}.parquet",
        tfidf_vocab_file=sample_dir / f"tfidf_vocabulary_{c['name']}.pkl",
        svd_components_file=sample_dir / f"svd_components_{c['name']}.pkl",
        feature_columns_file=sample_dir / f"feature_columns_{c['name']}.json",
    )

    pipe = FeaturePipeline(config=cfg)
    pipe.process_split_to_parquet([str(train_sample)], cfg.train_features_file, 'train')
    pipe.process_split_to_parquet([str(val_sample)], cfg.val_features_file, 'val')

    runner = RebaselineRunner(
        feature_config=cfg,
        model_config=RebaselineConfig(
            n_estimators=c['est'],
            max_depth=c['depth'],
            learning_rate=c['lr'],
        ),
    )
    summary = runner.run_full_rebaseline()
    rows.append({
        'name': c['name'],
        'accuracy': summary['full']['accuracy'],
        'macro_f1': summary['full']['macro_f1'],
        'feature_count': summary['full']['used_feature_count'],
    })

result_df = pd.DataFrame(rows).sort_values(['accuracy', 'macro_f1'], ascending=False)
result_df

,name,accuracy,macro_f1,feature_count
2,cfg_c_1to2_df1_deeper,0.383000,0.306062,177
0,cfg_a_1to2_df2,0.365000,0.307539,177
1,cfg_b_2to3_df1,0.356667,0.296512,177


In [9]:
# Lưu kết quả tuning
out_csv = sample_dir / 'tuning_iteration2_results.csv'
result_df.to_csv(out_csv, index=False)
out_csv

PosixPath('/home/sakana/Code/PTIT/MMDs/MMD-G2/data/features/smoke/tuning_iteration2_results.csv')

## Next
- Chọn config tốt nhất theo accuracy (tie-break bằng macro_f1).
- Cập nhật config mặc định trong `src/feature_config.py`.
- Chạy lại smoke notebook `rebaseline_smoke.ipynb` với config thắng.